<a href="https://colab.research.google.com/github/Matheus-200/python_sis_26_2/blob/Matheus_Gabriel/aula_11/matheus_gabriel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Atividade Prática 01: Gestão de Ativos e Auditoria de Logs

In [ ]:
# Etapa 1: Instalação, importação e criação da planilha
import openpyxl
from openpyxl.styles import PatternFill

wb = openpyxl.Workbook()
ws_servidores = wb.active
ws_servidores.title = "Servidores"

# Adicionando o cabeçalho
headers = ["Hostname", "IP", "Memória (GB)", "Uso de Disco (%)", "Status"]
ws_servidores.append(headers)

# Dados de exemplo para os servidores
dados_servidores = [
    ["srv-web-01", "192.168.1.10", 16, 75, "Ativo"],
    ["srv-db-01", "192.168.1.11", 32, 85, "Ativo"],
    ["srv-app-02", "192.168.1.12", 8, 45, "Ativo"],
    ["srv-cache-01", "192.168.1.13", 16, 90, "Ativo"]
]

for linha in dados_servidores:
    ws_servidores.append(linha)

# Etapa 2: Edição e Alerta de Armazenamento
fill_alerta = PatternFill(start_color="FFC7CE", end_color="FFC7CE", fill_type="solid") # Vermelho claro

for row in range(2, ws_servidores.max_row + 1):
    uso_disco = ws_servidores.cell(row=row, column=4).value
    hostname = ws_servidores.cell(row=row, column=1).value
    if uso_disco >= 80:
        print(f"⚠️ ALERTA: O servidor {hostname} está com {uso_disco}% de uso de disco e precisa de expansão!")
        # Colorindo a célula do Uso de Disco (%)
        ws_servidores.cell(row=row, column=4).fill = fill_alerta

# Criando o arquivo log.txt localmente de forma simples para garantir que ele exista
with open("log.txt", "w") as f:
    f.write("""2026-10-14 08:30:15 - INFO - Sistema inicializado com sucesso
2026-10-14 08:31:02 - INFO - Usuario 'admin' realizou login
2026-10-14 08:35:22 - AVISO - Tentativa de acesso com senha incorreta: usuario 'bat'
2026-10-14 08:35:40 - SUCESSO - Usuario 'bat' realizou login
2026-10-14 08:40:11 - ERRO - Falha ao conectar ao banco de dados: timeout
2026-10-14 08:42:00 - INFO - Conexao reestabelecida
""")

# Etapa 3 e 4: Extração do Log e salvando em nova aba de Auditoria
ws_logs = wb.create_sheet(title="Auditoria Logs")
ws_logs.append(["Data/Hora", "Tipo de Evento", "Mensagem"])

try:
    with open("log.txt", "r") as f:
        for linha in f:
            linha = linha.strip()
            if linha:
                # Exemplo de linha: 2026-10-14 08:30:15 - INFO - Sistema inicializado com sucesso
                partes = linha.split(" - ")
                if len(partes) >= 3:
                    data_hora = partes[0]
                    evento = partes[1]
                    mensagem = partes[2]
                    ws_logs.append([data_hora, evento, mensagem])
except FileNotFoundError:
    print("Arquivo log.txt não encontrado.")

# Salvando o arquivo final de auditoria no diretório
wb.save("auditoria_servidores.xlsx")
print("✔ Planilha 'auditoria_servidores.xlsx' gerada e salva com sucesso!")

### Atividade Prática 02: Coleta de Métricas de Hardware (Telemetria)

In [ ]:
import psutil
import time
import csv

# Definindo o arquivo e gravando o cabeçalho explicitamente
csv_path = "telemetria_sistema.csv"
with open(csv_path, "w", newline="") as f:
    f.write("Iteracao,Uso_CPU_Pct,Uso_RAM_Pct\n")

# Ciclo de amostragem: 5 iterações
print("Iniciando coleta de telemetria...")
for i in range(1, 6):
    # Captura da CPU (intervalo de 0.1s para precisão instantânea) e Memória
    cpu = psutil.cpu_percent(interval=0.1)
    ram = psutil.virtual_memory().percent

    print(f"Amostra {i}: CPU: {cpu}% | RAM: {ram}%")

    # Salvando incrementalmente no modo 'a'
    with open(csv_path, "a", newline="") as f:
        escritor = csv.writer(f)
        escritor.writerow([i, cpu, ram])

    # Pausa de 1 segundo (exceto na última iteração)
    if i < 5:
        time.sleep(1)

print("✔ Telemetria finalizada e salva em 'telemetria_sistema.csv'!")

### Atividade Prática 03: Análise e Visualização dos Dados de Telemetria

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Etapa 1: Ingestão de dados com Pandas
df = pd.read_csv("telemetria_sistema.csv")
display(df)

# Etapa 2: Cálculo das médias aritméticas
media_cpu = df["Uso_CPU_Pct"].mean()
media_ram = df["Uso_RAM_Pct"].mean()

print(f"\n--- Resumo Descritivo ---")
print(f"Média de Consumo de CPU: {media_cpu:.2f}%")
print(f"Média de Consumo de RAM: {media_ram:.2f}%")

# Etapa 3: Gráfico de Linha simples
plt.figure(figsize=(8, 4))
plt.plot(df["Iteracao"], df["Uso_CPU_Pct"], marker='o', label='CPU (%)', color='blue')
plt.plot(df["Iteracao"], df["Uso_RAM_Pct"], marker='s', label='RAM (%)', color='orange')

plt.title("Variação do Uso de CPU e RAM")
plt.xlabel("Iterações")
plt.ylabel("Uso (%)")
plt.xticks(df["Iteracao"])
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend()
plt.show()

### Resumo Simplificado das Atividades:
1. Criamos um inventário automatizado no Excel sinalizando servidores sobrecarregados e extraindo eventos importantes do arquivo de log.
2. Coletamos e registramos métricas reais de hardware do sistema segundo a segundo de forma incremental no formato CSV.
3. Analisamos a telemetria obtendo médias descritivas e plotamos gráficos de linha simples para o acompanhamento gerencial.